# D-algebraic functions expansion

In [1]:
import sys
sys.path.insert(0, "..") # dalgebra is here
from dalgebra import *
from dalgebra.pseries.laurent import *

%display latex

I want to think about what to do to detect possible orders for Laurent series expansions around 0 of solutions of D-algebraic equations. 

I am not fuly sure on all the steps but I have an intuition on how to proceed.

In order to work properly, I am going to need at least three examples:
* A linear example: things are easy here and a polynomial (indicial polynomial) can be computed.
* A purely D-algebraic case:
  - With guaranteed highest order monomial with its highest derivative
  - With highest order monomial without its highest derivative.
 
Then, it remains to check how to compute the expansion of the Laurent series solutions and how can we actually compute which inicial conditions are needed.

## Extending the series

In [36]:
def generate_coefficients_pseries(equation, gen, initials):
    initials = {initials[k] if k in initials else 0 for k in range(0, max(initials)+1)} # removing negative index
    order = equation.order(gen)
    if equation.degree(gen[order]) > 1:
        raise ValueError(f"Non-linear highest term")
    elif max(initials) < order-1:
        raise ValueError(f"Not enough initials")

    lc = equation.coefficient_full(gen[order])
    rhs = lc*gen[order] - equation

    LS_ring = LaurentSeries(equation.parent().constant_ring(), '_t')
    t = LS_ring.gen()

    def __get_value(k):
        global initials
        if k < min(initials):
            return 0
        elif k not in initials:## k not in initials: we compute and store it
            [__get_value(i) for i in range(k)] # we get previous values
            u_aux = sum(__get_value(i)/factorial(i)*t**i for i in range(k))
            dic_eval = {gen.variable_name() : u_aux}
            dic_eval.update({vname: 0 for vname in equation.parent().variable_names() if vname != gen.variable_name()})
            mor = equation.parent().laurent_morphism(dic_eval)
            n,d = mor(rhs), mor(lc)
            p = k-order
            initials[k] = (1/d[0])*(n[p]*factorial(p) - sum(u_aux[i]*factorial(i) * d[p-i]*factorial(p-i)*binomial(p,i) for i in range(p)))
        return initials[k]

## The linear case

In this case, we have an equation of the shape:
$$L = \alpha_0 u+ \alpha_1 u' + \ldots + \alpha_n u^{(n)}.$$
We assume the elements $\alpha_i \in C[[x]]$, so thei can only have a positive order.

Hence, each term will have order $d - i + ord(\alpha_i)$. This makes comparisons quite trivial and independent of $d$.

In [10]:
C = DifferentialRing(QQ)
R = DifferentialRing(QQ[x], [1])
R.set_constant(C)
F = R.fraction_field()
x, = F.gens()
DO.<u> = DifferentialPolynomialRing(F)
LS.<t> = LaurentSeries(C)
mor = F.laurent_morphism({'x': t}, set_default=True)

### 1.1. A monic case

$$L = u^{(5)} - 1/(1-x) u^{(3)} + 2xu'' - u$$

In [26]:
L = u[5] - (1/(1-x))*u[3] + 2*x*u[2] - u[0]
L
PR.<k> = PolynomialRing(ZZ)

In [27]:
def goal_order(mon,coeff,var):
    output = [0, None]
    for ((_,o),e) in mon._variables.items():
        output[0]+=e*var-o[0]
    output[0] += mor(coeff).order(bound=100)
    output[1] = mon
    return output

Here are the order for each term in the equation:

In [30]:
orders = [goal_order(m,c,k) for m,c in (zip(L.monomials(u), L.coefficients(u)))]

We would need that two of them are equal in order to have a cancellation. Hence:

In [32]:
candidates = []
for i in range(len(orders)):
    for j in range(i+1, len(orders)):
        ord_i = orders[i][0]
        ord_j = orders[j][0]
        candidates.append((ord_i - ord_j).roots())
candidates

[[], [], [], [], [], []]

Since we have zero candidates, then we conclude that the order of the series must be greater or equal than 0 and smaller than $5$.